In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt


from tqdm import tqdm

from relbench.base import TaskType
from relbench.tasks import get_task

from redelex.datasets import get_all_datasets_info

from notebooks.utils import get_potato_client, get_experiment_runs_df, get_run_metrics

from experiments.universal_encoder.universal_encoder_pretrained import PRETRAIN_TASKS

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 400)
pd.set_option("display.float_format", lambda x: "%.4f" % x)


%reload_ext autoreload
%autoreload 2

In [ ]:
def get_metrics(task_type: TaskType):
    if task_type == TaskType.REGRESSION:
        metrics = ["mae", "mse"]
    elif task_type == TaskType.BINARY_CLASSIFICATION:
        metrics = ["roc_auc", "accuracy"]
    else:
        raise ValueError(f"Unknown task type: {task_type}")
    return metrics

In [ ]:
experiment_names = [
    # "pelesjak_hetero_sage",
    "pelesjak_universal_pretrained_homoge_validation",
    "pelesjak_universal_pretrained_hetero_validation",
    "pelesjak_universal_pretrained_homoge_abla_validation",
    "pelesjak_universal_pretrained_homogeneous",
    "pelesjak_universal_pretrained_heterogeneous",
    "pelesjak_universal_pretrained_homogeneous_ablations",
]

client = get_potato_client()
for experiment_name in experiment_names:
    df = get_experiment_runs_df(client, experiment_name)
    df.to_csv(f"./data/{experiment_name}.csv", index=False)

In [ ]:
for experiment_name in experiment_names:
    df = pd.read_csv(f"./data/{experiment_name}.csv")
    if "validation" in experiment_name or "sage" in experiment_name:
        task_type_dict = {
            (dn, tn): get_task(dn, tn).task_type
            for dn, tn in df[["dataset_name", "task_name"]].drop_duplicates().values
        }
        task_type_metrics = {}
        for task_type in task_type_dict.values():
            metrics_df = get_metrics(task_type)
            task_type_metrics[task_type] = [f"val_{m}" for m in metrics_df] + [
                f"test_{m}" for m in metrics_df
            ]

    df_metrics = []
    _df_metrics = None
    try:
        _df_metrics = pd.read_csv(f"./data/{experiment_name}_metrics.csv")
    except FileNotFoundError:
        pass

    for i, r in tqdm(df.iterrows(), total=len(df)):
        if "validation" in experiment_name or "sage" in experiment_name:
            metrics_df = task_type_metrics[
                task_type_dict[(r["dataset_name"], r["task_name"])]
            ]
        else:
            metrics_df = ["train_loss_epoch", "val_loss_epoch"]
        if _df_metrics is not None and r["_run_id"] in _df_metrics["run_id"].values:
            continue
        run_metrics = get_run_metrics(client, r["_run_id"], metrics_df)
        if run_metrics is None:
            continue
        df_metrics.append(run_metrics)
        if i % 10 == 0:
            try:
                _df_metrics = pd.read_csv(f"./data/{experiment_name}_metrics.csv")
                _df_metrics = pd.concat([_df_metrics] + df_metrics).reset_index(drop=True)
            except FileNotFoundError:
                _df_metrics = pd.concat(df_metrics).reset_index(drop=True)
            _df_metrics.to_csv(f"./data/{experiment_name}_metrics.csv", index=False)
            df_metrics = []

    _df_metrics = pd.read_csv(f"./data/{experiment_name}_metrics.csv")
    _df_metrics = pd.concat([_df_metrics] + df_metrics).reset_index(drop=True)
    _df_metrics.to_csv(f"./data/{experiment_name}_metrics.csv", index=False)

In [ ]:
df_baseline = pd.read_csv("./data/pelesjak_hetero_sage.csv")
df_baseline_metrics = pd.read_csv("./data/pelesjak_hetero_sage_metrics.csv")
df_homo_train = pd.read_csv("./data/pelesjak_universal_pretrained_homogeneous.csv")
df_homo_train_metrics = pd.read_csv(
    "./data/pelesjak_universal_pretrained_homogeneous_metrics.csv"
)
df_hetero_train = pd.read_csv("./data/pelesjak_universal_pretrained_heterogeneous.csv")
df_hetero_train_metrics = pd.read_csv(
    "./data/pelesjak_universal_pretrained_heterogeneous_metrics.csv"
)
df_homo_abla_train = pd.read_csv(
    "./data/pelesjak_universal_pretrained_homogeneous_ablations.csv"
)
df_homo_abla_train_metrics = pd.read_csv(
    "./data/pelesjak_universal_pretrained_homogeneous_ablations_metrics.csv"
)

df_homo_val = pd.read_csv("./data/pelesjak_universal_pretrained_homoge_validation.csv")
df_hetero_val = pd.read_csv("./data/pelesjak_universal_pretrained_hetero_validation.csv")
df_homo_val_metrics = pd.read_csv(
    "./data/pelesjak_universal_pretrained_homoge_validation_metrics.csv"
)
df_hetero_val_metrics = pd.read_csv(
    "./data/pelesjak_universal_pretrained_hetero_validation_metrics.csv"
)
df_homo_abla_val = pd.read_csv(
    "./data/pelesjak_universal_pretrained_homoge_abla_validation.csv"
)
df_homo_abla_val_metrics = pd.read_csv(
    "./data/pelesjak_universal_pretrained_homoge_abla_validation_metrics.csv"
)

### Pretraining progress of Universal Row Encoder based on leftout dataset


In [ ]:
num_layers = df_homo_train["row_encoder_layers"].unique()
leave_out_datasets = df_homo_train["leave_out_dataset"].unique()

In [ ]:
# display(df_homo_train.columns[:100])


df_dict = {
    "With Shared Homogeneous GNN": (df_homo_train, df_homo_train_metrics),
    "With Task-specific Heterogeneous GNN": (df_hetero_train, df_hetero_train_metrics),
    # "With Shared Homogeneous GNN (Ablation)": (df_homo_abla_train, df_homo_abla_train_metrics),
}

r_ids = []

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 10), sharex="col")


for j, dataset in enumerate(leave_out_datasets):
    ax = axes[j // 3, j % 3]
    for i, (df_name, (df, df_metrics)) in enumerate(df_dict.items()):
        # r_ids = df["_run_id"].tolist()
        r_ids = []
        r_ids.extend(df[df["leave_out_dataset"] == dataset]["_run_id"].tolist())

        run_metrics: pd.DataFrame = df_metrics[df_metrics["run_id"].isin(r_ids)]
        run_metrics["step_bins"] = pd.cut(
            run_metrics["step"],
            bins=np.arange(0, 50000, 500),
            labels=np.arange(0, 49500, 500),
        )
        run_metrics = run_metrics.groupby("step_bins")[
            ["train_loss_epoch", "val_loss_epoch"]
        ].aggregate(["mean", "std"])
        run_metrics = run_metrics.ewm(span=5, adjust=False).mean()

        # plt.fill_between(
        #     run_metrics.index,
        #     run_metrics["train_loss_epoch"]["mean"]
        #     - run_metrics["train_loss_epoch"]["std"],
        #     run_metrics["train_loss_epoch"]["mean"]
        #     + run_metrics["train_loss_epoch"]["std"],
        #     alpha=0.4,
        # )
        ax.plot(
            run_metrics.index,
            run_metrics["val_loss_epoch"]["mean"],
            label=f"{df_name} - Val Loss",
            color=plt.get_cmap("tab10")(i + 8),
        )
        ax.plot(
            run_metrics.index,
            run_metrics["train_loss_epoch"]["mean"],
            color=plt.get_cmap("tab10")(i + 8),
            linestyle="--",
            label=f"{df_name} - Train Loss",
        )
        ax.fill_between(
            run_metrics.index,
            run_metrics["val_loss_epoch"]["mean"] - run_metrics["val_loss_epoch"]["std"],
            run_metrics["val_loss_epoch"]["mean"] + run_metrics["val_loss_epoch"]["std"],
            alpha=0.2,
            color=plt.get_cmap("tab10")(i + 8),
        )

    ax.set_title(f"{dataset}")
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.set_ylim(0.12, 0.37)
    ax.xaxis.set_major_locator(plt.MultipleLocator(10000))
    ax.xaxis.set_minor_locator(plt.MultipleLocator(2000))
    ax.yaxis.set_major_locator(plt.MultipleLocator(0.05))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(0.01))
    ax.grid(True, which="major", linewidth=0.5)


lines_labels = [ax.get_legend_handles_labels() for ax in fig.axes]
lines, labels = [sum(lol, []) for lol in zip(*lines_labels)]

unique_labels = labels[: len(df_dict) * 2]
unique_lines = lines[: len(df_dict) * 2]

fig.legend(
    unique_lines,
    unique_labels,
    loc="lower center",
    ncol=len(df_dict),
    prop={"size": 16},
    bbox_to_anchor=(0.5, -0.05),
)
fig.suptitle("", fontsize=16)

fig.align_xlabels()
fig.align_ylabels()
fig.tight_layout(h_pad=0)

- **hightlights that the unified pretraining optimization works**


In [ ]:
tasks_dict = {
    tuple(r): get_task(r[0], r[1])
    for r in df_homo_val[["dataset_name", "task_name"]].drop_duplicates().values
    if r[0] not in ["rel-amazon", "rel-f1", "rel-trial"]
}
task_types = set(task.task_type for task in tasks_dict.values())


def get_metrics_df(run_ids: list[str], df_metrics: pd.DataFrame, metric: str):
    metrics_df = df_metrics[df_metrics["run_id"].isin(run_ids)]
    metrics_df["step_bins"] = pd.cut(
        metrics_df["step"], bins=np.arange(0, 5001, 100), labels=np.arange(0, 5000, 100)
    )
    metrics_df = (
        metrics_df.groupby(["step_bins"])[[f"val_{metric}", f"test_{metric}"]]
        .aggregate(["mean", "std"])
        .ewm(span=5, adjust=False)
        .mean()
    )
    return metrics_df


task_type_task = {tt: [] for tt in task_types}
for (dt, tn), task in tasks_dict.items():
    task_type_task[task.task_type].append((dt, tn))

nrows = 2
ncols = 3
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 5 * nrows))


for i, ((dt, tn), task) in enumerate(
    sorted(tasks_dict.items(), key=lambda x: x[1].task_type.value)
):
    metrics = get_metrics(task.task_type)
    metric = metrics[0]

    ax = axes[i // ncols, i % ncols]

    df_task = df_homo_val[
        (df_homo_val["dataset_name"] == dt) & (df_homo_val["task_name"] == tn)
    ]

    df_task_hetero = df_hetero_val[
        (df_hetero_val["dataset_name"] == dt) & (df_hetero_val["task_name"] == tn)
    ]

    df_task_encoder_gnn = df_task[
        (
            df_task["pretrained_row_encoder"]
            & df_task["pretrained_gnn"]
            & (df_task["finetune_pretrained"] == False)
        )
    ]

    df_task_encoder = df_task[
        df_task["pretrained_row_encoder"]
        & (df_task["pretrained_gnn"] == False)
        & (df_task["finetune_pretrained"] == False)
    ]

    df_task_encoder_hetero = df_task_hetero[
        df_task_hetero["pretrained_row_encoder"]
        & (df_task_hetero["pretrained_gnn"] == False)
        & (df_task_hetero["finetune_pretrained"] == False)
    ]

    baseline = df_baseline[
        (df_baseline["dataset_name"] == dt)
        & (df_baseline["task_name"] == tn)
        & (df_baseline["tabular_encoder_layers"] == 1)
    ]

    encoder_gnn_metrics = get_metrics_df(
        df_task_encoder_gnn["_run_id"], df_homo_val_metrics, metric
    )

    ax.plot(
        encoder_gnn_metrics.index,
        encoder_gnn_metrics[f"test_{metric}"]["mean"],
        label="(1) Frozen Universal Encoder & GNN",
        color=plt.get_cmap("tab10")(0),
    )
    ax.fill_between(
        encoder_gnn_metrics.index,
        encoder_gnn_metrics[f"test_{metric}"]["mean"]
        - encoder_gnn_metrics[f"test_{metric}"]["std"],
        encoder_gnn_metrics[f"test_{metric}"]["mean"]
        + encoder_gnn_metrics[f"test_{metric}"]["std"],
        alpha=0.2,
        color=plt.get_cmap("tab10")(0),
    )

    encoder_metrics = get_metrics_df(
        df_task_encoder["_run_id"], df_homo_val_metrics, metric
    )

    ax.plot(
        encoder_metrics.index,
        encoder_metrics[f"test_{metric}"]["mean"],
        label="(2) Frozen Universal Encoder (Homogeneous)",
        color=plt.get_cmap("tab10")(1),
    )
    ax.fill_between(
        encoder_metrics.index,
        encoder_metrics[f"test_{metric}"]["mean"]
        - encoder_metrics[f"test_{metric}"]["std"],
        encoder_metrics[f"test_{metric}"]["mean"]
        + encoder_metrics[f"test_{metric}"]["std"],
        alpha=0.2,
        color=plt.get_cmap("tab10")(1),
    )

    encoder_hetero_metrics = get_metrics_df(
        df_task_encoder_hetero["_run_id"], df_hetero_val_metrics, metric
    )

    ax.plot(
        encoder_hetero_metrics.index,
        encoder_hetero_metrics[f"test_{metric}"]["mean"],
        label="(3) Frozen Universal Encoder (Heterogenous)",
        color=plt.get_cmap("tab10")(2),
    )
    ax.fill_between(
        encoder_hetero_metrics.index,
        encoder_hetero_metrics[f"test_{metric}"]["mean"]
        - encoder_hetero_metrics[f"test_{metric}"]["std"],
        encoder_hetero_metrics[f"test_{metric}"]["mean"]
        + encoder_hetero_metrics[f"test_{metric}"]["std"],
        alpha=0.2,
        color=plt.get_cmap("tab10")(2),
    )

    base_metrics = (
        df_baseline_metrics[
            (df_baseline_metrics["run_id"].isin(baseline["_run_id"]))
            & (df_baseline_metrics["step"] < 5000)
        ]
        .groupby(["step"])[[f"val_{metric}", f"test_{metric}"]]
        .mean()
    )

    first_base_val_metric = base_metrics.loc[base_metrics.index[0], f"val_{metric}"]
    first_base_test_metric = base_metrics.loc[base_metrics.index[0], f"test_{metric}"]

    if metric == "mae":
        best_base_step = base_metrics[f"val_{metric}"].idxmin()
        ax.set_ylim(first_base_test_metric - 0.02, first_base_test_metric + 0.1)
    else:
        best_base_step = base_metrics[f"val_{metric}"].idxmax()
        ax.set_ylim(first_base_test_metric - 0.1, first_base_test_metric + 0.02)

    best_base_val_metric = base_metrics.loc[best_base_step, f"val_{metric}"]
    best_base_test_metric = base_metrics.loc[best_base_step, f"test_{metric}"]

    ax.plot(
        [0, 5000],
        [best_base_test_metric, best_base_test_metric],
        color="black",
        linestyle="--",
        label="Specialized HGNN (5000 steps)",
    )

    ax.plot(
        [0, 5000],
        [first_base_test_metric, first_base_test_metric],
        color="gray",
        linestyle="--",
        label="Specialized HGNN (1000 steps)",
    )
    ax.set_title(f"{dt}: {tn}")
    ax.set_xlabel("Step")
    ax.set_ylabel(metric.upper().replace("_", " "))
    ax.xaxis.set_major_locator(plt.MultipleLocator(1000))
    ax.xaxis.set_minor_locator(plt.MultipleLocator(200))

    ax.yaxis.set_major_locator(plt.MultipleLocator(0.05))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(0.01))
    ax.grid(True, which="major", linewidth=0.5)

lines_labels = [ax.get_legend_handles_labels() for ax in fig.axes]
lines, labels = [sum(lol, []) for lol in zip(*lines_labels)]

unique_labels = labels[:5]
unique_lines = lines[:5]

fig.legend(
    unique_lines,
    unique_labels,
    loc="lower center",
    ncol=3,
    prop={"size": 16},
    bbox_to_anchor=(0.5, -0.05),
)
fig.suptitle("")
fig.align_xlabels()
fig.align_ylabels()
fig.tight_layout(h_pad=0)
fig.show()

In [ ]:
tasks_dict = {
    tuple(r): get_task(r[0], r[1])
    for r in df_homo_val[["dataset_name", "task_name"]].drop_duplicates().values
    if r[0] not in ["rel-amazon", "rel-f1", "rel-trial"]
}
task_types = set(task.task_type for task in tasks_dict.values())


def get_metrics_df(run_ids: list[str], df_metrics: pd.DataFrame, metric: str):
    metrics_df = df_metrics[df_metrics["run_id"].isin(run_ids)]
    metrics_df["step_bins"] = pd.cut(
        metrics_df["step"], bins=np.arange(0, 5001, 100), labels=np.arange(0, 5000, 100)
    )
    metrics_df = (
        metrics_df.groupby(["step_bins"])[[f"val_{metric}", f"test_{metric}"]]
        .aggregate(["mean", "std"])
        .ewm(span=5, adjust=False)
        .mean()
    )
    return metrics_df


task_type_task = {tt: [] for tt in task_types}
for (dt, tn), task in tasks_dict.items():
    task_type_task[task.task_type].append((dt, tn))

nrows = 2
ncols = 3
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 5 * nrows))


for i, ((dt, tn), task) in enumerate(
    sorted(tasks_dict.items(), key=lambda x: x[1].task_type.value)
):
    metrics = get_metrics(task.task_type)
    metric = metrics[0]

    ax = axes[i // ncols, i % ncols]

    df_task = df_homo_val[
        (df_homo_val["dataset_name"] == dt) & (df_homo_val["task_name"] == tn)
    ]

    df_task_hetero = df_hetero_val[
        (df_hetero_val["dataset_name"] == dt) & (df_hetero_val["task_name"] == tn)
    ]

    df_task_encoder = df_task[
        df_task["pretrained_row_encoder"]
        & (df_task["pretrained_gnn"] == False)
        & (df_task["finetune_pretrained"] == False)
    ]

    df_task_encoder_finetuned = df_task[
        df_task["pretrained_row_encoder"]
        & (df_task["pretrained_gnn"] == False)
        & (df_task["finetune_pretrained"] == True)
    ]

    df_task_hetero_encoder = df_task_hetero[
        df_task_hetero["pretrained_row_encoder"]
        & (df_task_hetero["pretrained_gnn"] == False)
        & (df_task_hetero["finetune_pretrained"] == False)
    ]

    df_task_hetero_encoder_finetuned = df_task_hetero[
        df_task_hetero["pretrained_row_encoder"]
        & (df_task_hetero["pretrained_gnn"] == False)
        & (df_task_hetero["finetune_pretrained"] == True)
    ]

    baseline = df_baseline[
        (df_baseline["dataset_name"] == dt)
        & (df_baseline["task_name"] == tn)
        & (df_baseline["tabular_encoder_layers"] == 1)
    ]

    encoder_metrics = get_metrics_df(
        df_task_encoder["_run_id"], df_homo_val_metrics, metric
    )

    ax.plot(
        encoder_metrics.index,
        encoder_metrics[f"test_{metric}"]["mean"],
        label="(1) Frozen Universal Encoder (Homogeneous)",
        color=plt.get_cmap("tab10")(1),
    )
    ax.fill_between(
        encoder_metrics.index,
        encoder_metrics[f"test_{metric}"]["mean"]
        - encoder_metrics[f"test_{metric}"]["std"],
        encoder_metrics[f"test_{metric}"]["mean"]
        + encoder_metrics[f"test_{metric}"]["std"],
        alpha=0.2,
        color=plt.get_cmap("tab10")(1),
    )

    encoder_metrics_finetuned = get_metrics_df(
        df_task_encoder_finetuned["_run_id"], df_homo_val_metrics, metric
    )

    ax.plot(
        encoder_metrics_finetuned.index,
        encoder_metrics_finetuned[f"test_{metric}"]["mean"],
        label="(2) Finetuned Universal Encoder (Homogeneous)",
        color=plt.get_cmap("tab10")(3),
    )
    ax.fill_between(
        encoder_metrics_finetuned.index,
        encoder_metrics_finetuned[f"test_{metric}"]["mean"]
        - encoder_metrics_finetuned[f"test_{metric}"]["std"],
        encoder_metrics_finetuned[f"test_{metric}"]["mean"]
        + encoder_metrics_finetuned[f"test_{metric}"]["std"],
        alpha=0.2,
        color=plt.get_cmap("tab10")(3),
    )

    encoder_hetero_metrics = get_metrics_df(
        df_task_hetero_encoder["_run_id"], df_hetero_val_metrics, metric
    )

    ax.plot(
        encoder_hetero_metrics.index,
        encoder_hetero_metrics[f"test_{metric}"]["mean"],
        label="(1) Frozen Universal Encoder (Heterogenous)",
        color=plt.get_cmap("tab10")(2),
    )
    ax.fill_between(
        encoder_hetero_metrics.index,
        encoder_hetero_metrics[f"test_{metric}"]["mean"]
        - encoder_hetero_metrics[f"test_{metric}"]["std"],
        encoder_hetero_metrics[f"test_{metric}"]["mean"]
        + encoder_hetero_metrics[f"test_{metric}"]["std"],
        alpha=0.2,
        color=plt.get_cmap("tab10")(2),
    )

    encoder_hetero_finetuned_metrics = get_metrics_df(
        df_task_hetero_encoder_finetuned["_run_id"], df_hetero_val_metrics, metric
    )

    ax.plot(
        encoder_hetero_finetuned_metrics.index,
        encoder_hetero_finetuned_metrics[f"test_{metric}"]["mean"],
        label="(2) Finetuned Universal Encoder (Heterogenous)",
        color=plt.get_cmap("tab10")(4),
    )
    ax.fill_between(
        encoder_hetero_finetuned_metrics.index,
        encoder_hetero_finetuned_metrics[f"test_{metric}"]["mean"]
        - encoder_hetero_finetuned_metrics[f"test_{metric}"]["std"],
        encoder_hetero_finetuned_metrics[f"test_{metric}"]["mean"]
        + encoder_hetero_finetuned_metrics[f"test_{metric}"]["std"],
        alpha=0.2,
        color=plt.get_cmap("tab10")(4),
    )

    base_metrics = (
        df_baseline_metrics[
            (df_baseline_metrics["run_id"].isin(baseline["_run_id"]))
            & (df_baseline_metrics["step"] < 5000)
        ]
        .groupby(["step"])[[f"val_{metric}", f"test_{metric}"]]
        .mean()
    )

    first_base_val_metric = base_metrics.loc[base_metrics.index[0], f"val_{metric}"]
    first_base_test_metric = base_metrics.loc[base_metrics.index[0], f"test_{metric}"]

    if metric == "mae":
        best_base_step = base_metrics[f"val_{metric}"].idxmin()
        ax.set_ylim(first_base_test_metric - 0.02, first_base_test_metric + 0.1)
    else:
        best_base_step = base_metrics[f"val_{metric}"].idxmax()
        ax.set_ylim(first_base_test_metric - 0.1, first_base_test_metric + 0.02)

    best_base_val_metric = base_metrics.loc[best_base_step, f"val_{metric}"]
    best_base_test_metric = base_metrics.loc[best_base_step, f"test_{metric}"]

    ax.plot(
        [0, 5000],
        [best_base_test_metric, best_base_test_metric],
        color="black",
        linestyle="--",
        label="Specialized HGNN (5000 steps)",
    )

    ax.plot(
        [0, 5000],
        [first_base_test_metric, first_base_test_metric],
        color="gray",
        linestyle="--",
        label="Specialized HGNN (1000 steps)",
    )
    ax.set_title(f"{dt}: {tn}")
    ax.set_xlabel("Step")
    ax.set_ylabel(metric.upper().replace("_", " "))
    ax.xaxis.set_major_locator(plt.MultipleLocator(1000))
    ax.xaxis.set_minor_locator(plt.MultipleLocator(200))

    ax.yaxis.set_major_locator(plt.MultipleLocator(0.05))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(0.01))
    ax.grid(True, which="major", linewidth=0.5)

lines_labels = [ax.get_legend_handles_labels() for ax in fig.axes]
lines, labels = [sum(lol, []) for lol in zip(*lines_labels)]

unique_labels = labels[:6]
unique_lines = lines[:6]

fig.legend(
    unique_lines,
    unique_labels,
    loc="lower center",
    ncol=3,
    prop={"size": 16},
    bbox_to_anchor=(0.5, -0.05),
)
fig.suptitle("")
fig.align_xlabels()
fig.align_ylabels()
fig.tight_layout(h_pad=0)
fig.show()

### Model Size and Performance Table


In [ ]:
tasks_dict = {
    tuple(r): get_task(r[0], r[1])
    for r in df_homo_val[["dataset_name", "task_name"]].drop_duplicates().values
}
task_types = set(task.task_type for task in tasks_dict.values())


task_type_task = {tt: [] for tt in task_types}
for (dt, tn), task in tasks_dict.items():
    task_type_task[task.task_type].append((dt, tn))


for task_type in task_type_task:
    metrics = get_metrics(task_type)
    metric = metrics[0]

    df = df_baseline[
        (df_baseline["tabular_encoder_layers"] == 4)
        & (df_baseline[[f"best_val_{metric}", f"best_test_{metric}"]].notnull().all(axis=1))
    ]

    group_by = ["dataset_name", "task_name"]

    if metric == "mae":
        idx_df = df.groupby(group_by)[f"best_val_{metric}"].idxmin()
    elif metric == "roc_auc":
        idx_df = df.groupby(group_by)[f"best_val_{metric}"].idxmax()

    specialized_df = (
        df.loc[idx_df][
            [
                *group_by,
                f"best_val_{metric}",
                f"best_test_{metric}",
                "model_size_MB",
            ]
        ]
        .set_index(group_by, drop=True)
        .sort_index()
    )

    df = df_homo_val[
        df_homo_val["pretrained_row_encoder"]
        & (df_homo_val["pretrained_gnn"] == True)
        & (df_homo_val["finetune_pretrained"] == False)
        & (df_homo_val[[f"best_val_{metric}", f"best_test_{metric}"]].notnull().all(axis=1))
    ]

    if metric == "mae":
        idx_df = df.groupby(group_by)[f"best_val_{metric}"].idxmin()
    elif metric == "roc_auc":
        idx_df = df.groupby(group_by)[f"best_val_{metric}"].idxmax()

    pretrained_df = (
        df.loc[idx_df][
            [
                *group_by,
                f"best_val_{metric}",
                f"best_test_{metric}",
                "model_size_MB",
            ]
        ]
        .set_index(group_by, drop=True)
        .sort_index()
    )
    metric_cols = [f"best_val_{metric}", f"best_test_{metric}"]
    # pretrained_df[metric_cols] = pretrained_df[metric_cols] - specialized_df[metric_cols]

    pretrained_df = pretrained_df.reindex(specialized_df.index)

    specialized_df_val = specialized_df.copy(deep=True)
    specialized_df_test = specialized_df.copy(deep=True)

    specialized_df_val["split"] = "val"
    specialized_df_val = specialized_df_val.drop(columns=[f"best_test_{metric}"])
    specialized_df_val.rename(columns={f"best_val_{metric}": metric}, inplace=True)
    specialized_df_test["split"] = "test"
    specialized_df_test = specialized_df_test.drop(columns=[f"best_val_{metric}"])
    specialized_df_test.rename(columns={f"best_test_{metric}": metric}, inplace=True)

    specialized_df = pd.concat(
        [specialized_df_val, specialized_df_test], axis=0
    ).reset_index()

    pretrained_df_val = pretrained_df.copy(deep=True)
    pretrained_df_test = pretrained_df.copy(deep=True)
    pretrained_df_val["split"] = "val"
    pretrained_df_val.rename(columns={f"best_val_{metric}": metric}, inplace=True)
    pretrained_df_val = pretrained_df_val.drop(columns=[f"best_test_{metric}"])
    pretrained_df_test["split"] = "test"
    pretrained_df_test = pretrained_df_test.drop(columns=[f"best_val_{metric}"])
    pretrained_df_test.rename(columns={f"best_test_{metric}": metric}, inplace=True)

    pretrained_df = pd.concat([pretrained_df_val, pretrained_df_test], axis=0).reset_index()

    df_compare: pd.DataFrame = pd.concat(
        [specialized_df, pretrained_df], axis=1, keys=["Specialized", "Universal"]
    )

    df_compare = df_compare.set_index(
        [
            ("Specialized", "dataset_name"),
            ("Specialized", "task_name"),
            ("Specialized", "split"),
        ]
    )

    df_compare.index.names = ["dataset_name", "task_name", "split"]
    df_compare = df_compare.drop(
        columns=["dataset_name", "task_name", "split"], level=1
    ).sort_index(level=[0, 1, 2], ascending=[True, True, False])

    display(df_compare)

    df_compare.to_latex(
        f"./figures/{task_type.value}_compare.tex",
        float_format="%.2f",
        escape=True,
        index=True,
        multicolumn=True,
        multirow=True,
    )

In [ ]:
df_info = get_all_datasets_info()

df_info: pd.DataFrame = df_info[df_info["dataset"].isin(PRETRAIN_TASKS.keys())]
df_info: pd.DataFrame = df_info[
    [
        "dataset",
        "domain",
        "n_tables",
        "n_fks",
        "n_factual_cols",
        "schema_diameter",
        "has_loops",
        "one_to_one",
        "one_to_many",
    ]
].sort_values(by="dataset")

df_info.to_latex("./figures/db-info.tex", index=False)